In [ ]:

import pandas as pd

# 读取训练数据
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/crime_category/train.csv')

# 读取测试数据
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/crime_category/test.csv')

# 查看数据的前几行
print(train_data.head())
print(test_data.head())


        Category  DayOfWeek PdDistrict           X          Y
0  LARCENY/THEFT  Wednesday   SOUTHERN -122.388380  37.783310
1   NON-CRIMINAL   Saturday   SOUTHERN -122.403405  37.775421
2  LARCENY/THEFT  Wednesday   NORTHERN -122.419581  37.789214
3  VEHICLE THEFT     Friday    BAYVIEW -122.389744  37.757909
4        ASSAULT     Friday    TARAVAL -122.478377  37.742877
         Category  DayOfWeek  PdDistrict           X          Y
0   VEHICLE THEFT     Sunday    SOUTHERN -122.409893  37.780113
1        BURGLARY     Monday    NORTHERN -122.424442  37.788227
2  OTHER OFFENSES  Wednesday     BAYVIEW -122.395635  37.753565
3        BURGLARY    Tuesday        PARK -122.444802  37.754171
4    NON-CRIMINAL   Saturday  TENDERLOIN -122.412971  37.785788


In [ ]:


# Encode categorical variables
train_data_encoded = pd.get_dummies(train_data, columns=['DayOfWeek', 'PdDistrict', 'Category'])
test_data_encoded = pd.get_dummies(test_data, columns=['DayOfWeek', 'PdDistrict'])

# Ensure test data has the same columns as train data
train_columns = train_data_encoded.columns
test_data_encoded = test_data_encoded.reindex(columns=train_columns, fill_value=0)

# Split the data into features and target variables
X_train = train_data_encoded.drop('Category_Assault', axis=1)
y_train = train_data_encoded['Category_Assault']
X_test = test_data_encoded.drop('Category_Assault', axis=1)

# Print shapes to verify
print(X_train.shape, y_train.shape)
print(X_test.shape)



---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
Cell In[3], line 20
     18 pass
     19 print('##active_line10##')
---> 20 X_train = train_data_encoded.drop('Category_Assault', axis=1)
     21 print('##active_line11##')
     22 y_train = train_data_encoded['Category_Assault']

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\frame.py:5581, in DataFrame.drop(self, labels, axis, index, columns, level, inplace, errors)
   5433 def drop(
   5434     self,
   5435     labels: IndexLabel | None = None,
   (...)
   5442     errors: IgnoreRaise = "raise",
   5443 ) -> DataFrame | None:
   5444     """
   5445     Drop specified labels from rows or columns.
   5446 
   (...)
   5579             weight  1.0     0.8
   5580     """
-> 5581     return super().drop(
   5582         labels=labels,
   5583         axis=axis,
   5584         index=

In [ ]:

# Encode categorical variables
train_data_encoded = pd.get_dummies(train_data, columns=['DayOfWeek', 'PdDistrict', 'Category'])
test_data_encoded = pd.get_dummies(test_data, columns=['DayOfWeek', 'PdDistrict'])

# Ensure test data has the same columns as train data
train_columns = train_data_encoded.columns
test_data_encoded = test_data_encoded.reindex(columns=train_columns, fill_value=0)

# List of all target categories
target_categories = [col for col in train_data_encoded if col.startswith('Category_')]

# Split the data into features and target variables
X_train = train_data_encoded.drop(target_categories, axis=1)
y_train = train_data_encoded[target_categories]
X_test = test_data_encoded.drop(target_categories, axis=1)
y_test = test_data_encoded[target_categories]

# Print shapes to verify
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(70244, 19) (70244, 37)
(17560, 19) (17560, 37)


In [ ]:


# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
import numpy as np

# Initialize the model
model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))

# Train the model
model.fit(X_train, y_train)




Displayed on the user's machine.

In [ ]:


# Import necessary libraries
from sklearn.metrics import log_loss

# Make predictions on the test data
y_pred_proba = model.predict_proba(X_test)

# Calculate Log Loss
logloss = log_loss(y_test, y_pred_proba, labels=y_train.columns.tolist())

# Print the Log Loss
print(f"Log Loss: {logloss:.4f}")

# Save predictions to a CSV file
predictions_df = pd.DataFrame(y_pred_proba, columns=target_categories)
predictions_df.to_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/crime_category/predictions.csv', index=False)

# Print the first few rows of the predictions
print(predictions_df.head())



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_param_validation.py:216, in validate_params.<locals>.decorator.<locals>.wrapper(*args, **kwargs)
    210 try:
    211     with config_context(
    212         skip_parameter_validation=(
    213             prefer_skip_nested_validation or global_skip_validation
    214         )
    215     ):
--> 216         return func(*args, **kwargs)
    217 except InvalidParameterError as e:
    218     # When the function is just a wrapper around an estimator, we allow
    219     # the function to delegate validation to the estimator, but we replace
    220     # the name of the estimator by the name of the function in the error
    221     # message to avoid confusion.
    222     msg = re.sub(
    223         r"parameter of \w+ must be",
    224     

In [ ]:



# Import necessary libraries
from sklearn.metrics import log_loss

# Make predictions on the test data
y_pred_proba = model.predict_proba(X_test)

# Reshape y_pred_proba to a 2D array
y_pred_proba_2d = np.array([prob for sublist in y_pred_proba for prob in sublist])

# Reshape y_test to a 1D array
y_test_1d = y_test.values.ravel()

# Ensure y_test and y_pred_proba have the same number of samples
if len(y_test_1d) != len(y_pred_proba_2d):
    min_length = min(len(y_test_1d), len(y_pred_proba_2d))
    y_test_1d = y_test_1d[:min_length]
    y_pred_proba_2d = y_pred_proba_2d[:min_length]

# Calculate Log Loss
logloss = log_loss(y_test_1d, y_pred_proba_2d)

# Print the Log Loss
print(f"Log Loss: {logloss:.4f}")

# Save predictions to a CSV file
predictions_df = pd.DataFrame(y_pred_proba_2d, columns=['predicted_prob'])
predictions_df.to_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/crime_category/predictions.csv', index=False)

# Print the first few rows of the predictions
print(predictions_df.head())




---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[7], line 40
     38 pass
     39 print('##active_line20##')
---> 40 logloss = log_loss(y_test_1d, y_pred_proba_2d)
     41 print('##active_line21##')
     42 pass

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_param_validation.py:216, in validate_params.<locals>.decorator.<locals>.wrapper(*args, **kwargs)
    210 try:
    211     with config_context(
    212         skip_parameter_validation=(
    213             prefer_skip_nested_validation or global_skip_validation
    214         )
    215     ):
--> 216         return func(*args, **kwargs)
    217 except InvalidParameterError as e:
    218     # When the function is just a wrapper around an estimator, we allow
    219     # the function to delegate validation to the estimator, but we replace
    220     # the name of 